#Imports and setup

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 31.6 MB/s eta 0:00:00


In [10]:
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

DATA_PATH_POLITIK    = Path('/content/drive/MyDrive/CSS_Analyse/reddit_politik_raw.csv')
DATA_PATH_POLITIKBRD = Path('/content/drive/MyDrive/CSS_Analyse/reddit_politikbrd_raw.csv')

print(f'Data 1: {DATA_PATH_POLITIK}')
print(f'Data 2: {DATA_PATH_POLITIKBRD}')

df_politik    = pd.read_csv(DATA_PATH_POLITIK)
df_politikbrd = pd.read_csv(DATA_PATH_POLITIKBRD)

print(f'r/politik:    {len(df_politik):,} Zeilen')
print(f'r/PolitikBRD: {len(df_politikbrd):,} Zeilen')

df = pd.concat([df_politik, df_politikbrd], ignore_index=True)

vor = len(df)
df = df.drop_duplicates(subset='id').reset_index(drop=True)
if vor != len(df):
    print(f'{vor - len(df)} Duplikate entfernt')

df['created_utc'] = pd.to_datetime(df['created_utc'], errors='coerce')
df = df.sort_values('created_utc').reset_index(drop=True)

print(f'\nKombinierter Datensatz: {len(df):,} Zeilen')
print(df['subreddit'].value_counts())
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data 1: /content/drive/MyDrive/CSS_Analyse/reddit_politik_raw.csv
Data 2: /content/drive/MyDrive/CSS_Analyse/reddit_politikbrd_raw.csv
r/politik:    398 Zeilen
r/PolitikBRD: 133 Zeilen

Kombinierter Datensatz: 531 Zeilen
subreddit
politik       398
PolitikBRD    133
Name: count, dtype: int64


,id,subreddit,title,body,text,word_count,score,num_comments,created_utc,flair
0,1hznmea,politik,"Machen die Linken nicht selbst genau das, was ...",Wurde zuletzt von einem Freund der bei der Ant...,"Machen die Linken nicht selbst genau das, was ...",48,0,60,2025-01-12,Frage
1,1hzm83e,politik,Wieso sind Leute voreingenommen? (AfD),NaN,Wieso sind Leute voreingenommen? (AfD),5,1,1,2025-01-12,Frage
2,1hzp9k4,politik,Warum ich die AfD verstehe und die Linke einfa...,Ich bin ehrlich gesagt ziemlich fertig mit dem...,Warum ich die AfD verstehe und die Linke einfa...,267,0,29,2025-01-12,Frage
3,1hzpb96,politik,"Wie passiert es eigentlich, dass Menschen ande...","Ich habe leider keinen anderen Kanal gefunden,...","Wie passiert es eigentlich, dass Menschen ande...",178,1,3,2025-01-12,Frage
4,1hzsv9v,politik,"Wann zahlen Boomer den Preis für das, was sie ...","Marode Infrastruktur, eine demographische Kata...","Wann zahlen Boomer den Preis für das, was sie ...",33,0,43,2025-01-12,Frage


#Candidates

In [11]:
import re
from collections import Counter

# ── Personen: Vor- und Nachname ━━━━━━━━━━━━━━━━━━━━━━━━
PERSONS = {
    'Olaf Scholz':     ('Olaf', 'Scholz'),
    'Friedrich Merz':  ('Friedrich', 'Merz'),
    'Alice Weidel':    ('Alice', 'Weidel'),
    'Robert Habeck':   ('Robert', 'Habeck'),
}

# ── Regex pro Person: Vor- + Nachname zusammen zählt nur 1x ━━━
# Reihenfolge der Alternativen ist wichtig: "Vorname Nachname" wird
# zuerst geprüft, sodass bei gemeinsamem Vorkommen nur EIN Treffer
# gezählt wird (statt je einem für Vor- und Nachname einzeln).
patterns = {}
for person, (first, last) in PERSONS.items():
    patterns[person] = re.compile(
        rf'\b{re.escape(first.lower())}\s+{re.escape(last.lower())}\b'
        rf'|\b{re.escape(first.lower())}\b'
        rf'|\b{re.escape(last.lower())}\b'
    )

# ── Zählung über alle Beiträge (df['text']) ━━━━━━━━━━━━
counts = Counter()

for text in df['text'].dropna():
    text_lower = text.lower()
    for person, pattern in patterns.items():
        counts[person] += len(pattern.findall(text_lower))

# ── Ausgabe ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Erwähnungen pro Person:\n")
for person in PERSONS:
    print(f"{person:18s}: {counts[person]:,}")

print(f"\nGesamt: {sum(counts.values()):,}")

Erwähnungen pro Person:

Olaf Scholz       : 41
Friedrich Merz    : 64
Alice Weidel      : 16
Robert Habeck     : 26

Gesamt: 147


#Issues

In [12]:
import re
from collections import Counter

# ── Themen mit ihren Synonymen/Begriffen ━━━━━━━━━━━━━━━
TOPICS = {
    'Wirtschaft': ['Wirtschaft', 'Inflation', 'Infrastruktur'],
    'Migration':  ['Migration', 'Zuwanderung',
                   'Migrant', 'Asyl'],
    'Aussenpolitik':  ['Krieg', 'Ausenpolitik','Konflikt',
                   'Außenpolitik'],
}

# ── Regex pro Begriff bauen (mehrwortfähig, z.B. "bewaffneter Konflikt") ━
term_patterns = {}
for topic, terms in TOPICS.items():
    for term in terms:
        escaped = re.escape(term.lower()).replace(r'\ ', r'\s+')
        term_patterns[(topic, term)] = re.compile(rf'\b{escaped}\b')

# ── Zählung über alle Beiträge (df['text']) ━━━━━━━━━━━━
topic_counts = Counter()
term_counts  = Counter()

for text in df['text'].dropna():
    text_lower = text.lower()
    for (topic, term), pattern in term_patterns.items():
        n = len(pattern.findall(text_lower))
        if n:
            term_counts[(topic, term)] += n
            topic_counts[topic] += n

# ── Ausgabe: Gesamt pro Thema ━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Erwähnungen pro Thema:\n")
for topic in TOPICS:
    print(f"{topic:15s}: {topic_counts[topic]:,}")
print(f"\nGesamt: {sum(topic_counts.values()):,}")

# ── Ausgabe: Aufschlüsselung pro Einzelbegriff ━━━━━━━━━
print("\n\nAufschlüsselung pro Begriff:\n")
for topic, terms in TOPICS.items():
    print(f"── {topic} ──")
    for term in terms:
        print(f"   {term:22s}: {term_counts[(topic, term)]:,}")

Erwähnungen pro Thema:

Wirtschaft     : 118
Migration      : 79
Aussenpolitik  : 31

Gesamt: 228


Aufschlüsselung pro Begriff:

── Wirtschaft ──
   Wirtschaft            : 91
   Inflation             : 19
   Infrastruktur         : 8
── Migration ──
   Migration             : 63
   Zuwanderung           : 5
   Migrant               : 2
   Asyl                  : 9
── Aussenpolitik ──
   Krieg                 : 23
   Ausenpolitik          : 0
   Konflikt              : 3
   Außenpolitik          : 5


#Candidates and topics

In [13]:
import re
import pandas as pd
from pathlib import Path

# ── Kandidaten: Vor- und Nachname ━━━━━━━━━━━━━━━━━━━━━━
PERSONS = {
    'Olaf Scholz':     ('Olaf', 'Scholz'),
    'Friedrich Merz':  ('Friedrich', 'Merz'),
    'Alice Weidel':    ('Alice', 'Weidel'),
    'Robert Habeck':   ('Robert', 'Habeck'),
}

# ── Themen mit ihren Synonymen/Begriffen ━━━━━━━━━━━━━━━
TOPICS = {
    'Wirtschaft': ['Wirtschaft', 'Inflation', 'Infrastruktur'],
    'Migration':  ['Migration', 'Zuwanderung',
                   'Migrant', 'Asyl'],
    'Aussenpolitik': ['Krieg', 'Ausenpolitik','Konflikt',
                   'Außenpolitik'],
}

# ── Regex pro Kandidat (Vor- + Nachname zusammen = 1 Treffer) ━
person_patterns = {}
for person, (first, last) in PERSONS.items():
    person_patterns[person] = re.compile(
        rf'\b{re.escape(first.lower())}\s+{re.escape(last.lower())}\b'
        rf'|\b{re.escape(first.lower())}\b'
        rf'|\b{re.escape(last.lower())}\b'
    )

# ── Regex pro Thema (alle Synonyme zu einem Pattern zusammengefasst) ━
topic_patterns = {}
for topic, terms in TOPICS.items():
    escaped_terms = [re.escape(t.lower()).replace(r'\ ', r'\s+') for t in terms]
    topic_patterns[topic] = re.compile(r'\b(?:' + '|'.join(escaped_terms) + r')\b')

# ── Für jeden Post prüfen: welcher Kandidat + welches Thema kommt vor ━
co_counts = pd.DataFrame(0, index=PERSONS.keys(), columns=TOPICS.keys())

for text in df['text'].dropna():
    text_lower = text.lower()

    persons_in_post = [p for p, pat in person_patterns.items() if pat.search(text_lower)]
    topics_in_post  = [t for t, pat in topic_patterns.items() if pat.search(text_lower)]

    for person in persons_in_post:
        for topic in topics_in_post:
            co_counts.loc[person, topic] += 1

# ── Ausgabe ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Kandidat × Thema — Anzahl gemeinsamer Erwähnungen in einem Post:\n")
print(co_counts.to_string())

print(f"\nGesamt (alle Kombinationen): {co_counts.values.sum():,}")

# Optional: als CSV speichern
out_path = Path('/content') / 'kandidat_thema_matrix.csv'
co_counts.to_csv(out_path)
print(f"\nMatrix gespeichert unter: {out_path}")


Kandidat × Thema — Anzahl gemeinsamer Erwähnungen in einem Post:

                Wirtschaft  Migration  Aussenpolitik
Olaf Scholz              4          3              1
Friedrich Merz           4         10              3
Alice Weidel             0          0              1
Robert Habeck            4          0              1

Gesamt (alle Kombinationen): 31

Matrix gespeichert unter: /content/kandidat_thema_matrix.csv
